<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="200" align="right" style="height:240px">

##ColabFold v1.6.1: AlphaFold2 using MMseqs2

Easy to use protein structure and complex prediction using [AlphaFold2](https://www.nature.com/articles/s41586-021-03819-2) and [Alphafold2-multimer](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1). Sequence alignments/templates are generated through [MMseqs2](mmseqs.com) and [HHsearch](https://github.com/soedinglab/hh-suite). For more details, see <a href="#Instructions">bottom</a> of the notebook, checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold) and [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5).

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/AlphaFold2.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/AlphaFold2.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/AlphaFold2.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/AlphaFold2.ipynb)

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

In [ ]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import random

from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]

#@markdown ### 📂 1. 분석 데이터 입력 방식 선택
INPUT_METHOD = "File_Upload" #@param ["Text_Input", "File_Upload"]
#@markdown - **File_Upload**: 실행 시 나오는 '파일 선택' 버튼으로 .fasta, .faa 파일 업로드
#@markdown - **Text_Input**: 아래 query_sequence 박스에 아미노산 서열을 입력

query_sequence = '' #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a homodimer
#@markdown  - **단백질 복합체를 모델링**할 때는 서로 다른 체인을 구분하기 위해 `:` 를 사용하세요 (동일 단백질로 이루어진 호모 올리고머와 서로 다른 단백질로 이루어진 헤테로 올리고머 모두에 적용)
#@markdown  - 예를 들어 **PI...SK:PI...SK**는 동일한 단백질 두 개로 구성된 호모다이머를 의미합니다

#@markdown ---

#@markdown ### ⚙️ 2. 분석 옵션 설정
jobname = '' #@param {type:"string"}
#@markdown - 작업명, 예: test
# number of models to use
num_relax = 1 #@param [0, 1, 5] {type:"raw"}
#@markdown - 예측된 구조 중 몇 개를 추가로 정제 할지를 정하는 값
#@markdown - specify how many of the top ranked structures to relax using amber
#@markdown - 상위 랭킹 구조들 중에서 AMBER를 사용해 구조를 정제할 개수를 지정하세요
template_mode = "pdb100" #@param ["none", "pdb100","custom"]
#@markdown - 템플릿 모드: 이미 알려진 단백질 구조 중 유사한 것을 참고하는 기능
#@markdown - `none` = no template information is used
#@markdown - `pdb100` = detect templates in pdb100 (see [notes](#pdb100))
#@markdown - `custom` - upload and search own templates (PDB or mmCIF format, see [notes](#custom_templates))

use_amber = num_relax > 0

# ==========================================
# INPUT_METHOD에 따른 서열 입력 처리
# ==========================================
if INPUT_METHOD == "File_Upload":
  print("단백질 서열 파일(.fasta, .faa)을 업로드해주세요.")
  uploaded_seq = files.upload()
  for fn in uploaded_seq.keys():
    with open(fn, 'r') as f:
      # FASTA 포맷 처리: '>'로 시작하는 헤더 줄은 제외하고 아미노산 서열만 추출
      lines = f.readlines()
      seqs = [line.strip() for line in lines if not line.startswith('>')]
      query_sequence = "".join(seqs)
    break  # 일단 첫 번째 업로드된 파일만 처리하도록 설정

# 서열이 비어있는지 확인 (에러 방지)
if not query_sequence or query_sequence.strip() == '':
  raise ValueError("🚨 오류: 아미노산 서열이 비어 있습니다. 텍스트를 입력하거나 파일을 정상적으로 업로드해주세요.")
# ==========================================

# remove whitespaces
query_sequence = "".join(query_sequence.split())

basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)

# check if directory with jobname exists
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True
if not check(jobname):
  n = 0
  while not check(f"{jobname}_{n}"): n += 1
  jobname = f"{jobname}_{n}"

# make directory to save results
os.makedirs(jobname, exist_ok=True)

# save queries
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
  text_file.write(f"id,sequence\n{jobname},{query_sequence}")

if template_mode == "pdb100":
  use_templates = True
  custom_template_path = None
elif template_mode == "custom":
  custom_template_path = os.path.join(jobname,f"template")
  os.makedirs(custom_template_path, exist_ok=True)
  uploaded = files.upload()
  use_templates = True
  for fn in uploaded.keys():
    os.rename(fn,os.path.join(custom_template_path,fn))
else:
  custom_template_path = None
  use_templates = False

print("jobname",jobname)
print("sequence",query_sequence)
print("length",len(query_sequence.replace(":","")))

In [ ]:
#@title Install dependencies
%%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # hack to fix TF crash
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("installing hhsuite and amber...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("installing hhsuite...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("installing amber...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

In [ ]:
#@markdown ### MSA(Multiple Sequence Alignment) 생성 옵션
#@markdown ### MSA options (custom MSA upload, single sequence, pairing mode)
msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref","single_sequence","custom"]
#@markdown - 기본값인 MMseqs2(UniRef+Environmental)는 UniRef 데이터베이스뿐 아니라 환경 유래 서열도 함께 참고해서, 진화 정보가 부족한 단백질도 어느 정도 커버할 수 있도록 도움
pair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}
#@markdown - "unpaired_paired" = pair sequences from same species + unpaired MSA
#@markdown - "unpaired" = seperate MSA for each chain
#@markdown - "paired" - only use paired sequences

# decide which a3m to use
if "mmseqs2" in msa_mode:
  a3m_file = os.path.join(jobname,f"{jobname}.a3m")

elif msa_mode == "custom":
  a3m_file = os.path.join(jobname,f"{jobname}.custom.a3m")
  if not os.path.isfile(a3m_file):
    custom_msa_dict = files.upload()
    custom_msa = list(custom_msa_dict.keys())[0]
    header = 0
    import fileinput
    for line in fileinput.FileInput(custom_msa,inplace=1):
      if line.startswith(">"):
         header = header + 1
      if not line.rstrip():
        continue
      if line.startswith(">") == False and header == 1:
         query_sequence = line.rstrip()
      print(line, end='')

    os.rename(custom_msa, a3m_file)
    queries_path=a3m_file
    print(f"moving {custom_msa} to {a3m_file}")

else:
  a3m_file = os.path.join(jobname,f"{jobname}.single_sequence.a3m")
  with open(a3m_file, "w") as text_file:
    text_file.write(">1\n%s" % query_sequence)

In [ ]:
#@markdown ### Advanced settings
model_type = "auto" #@param ["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3", "deepfold_v1", "alphafold2"]
#@markdown - model_type 옵션: 어떤 모델을 사용할지 결정하는 설정
#@markdown - if `auto` selected, will use `alphafold2_ptm` for monomer prediction and `alphafold2_multimer_v3` for complex prediction
#@markdown - Any of the mode_types can be used (regardless if input is monomer or complex)
#@markdown - 기본값 = `auto` - 입력한 서열에 따라 자동으로 가장 적절한 모델 적용
#@markdown - 단일 사슬(mononer) 구조일 경우: `alphafold2_ptm` 모델 사용
#@markdown - 복합체(multimer) 구조일 경우: `alphafold2_multimer_v3` 모델 사용

num_recycles = "auto" #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
#@markdown - 예측된 구조를 몇 번 반복해서 다듬을지 정하는 값
#@markdown - if `auto` selected, will use `num_recycles=20` if `model_type=alphafold2_multimer_v3`, else `num_recycles=3`

recycle_early_stop_tolerance = "auto" #@param ["auto", "0.0", "0.5", "1.0"]
#@markdown - if `auto` selected, will use `tol=0.5` if `model_type=alphafold2_multimer_v3` else `tol=0.0`
#@markdown - 예측 결과를 반복적으로 다듬는 과정을 언제 멈출지 결정하는 설정
#@markdown - 기본값 = `auto` - 단일 단백질(monomer)이면 0.0, 복합체(multimer_v3)면 0.5로 자동 적용

relax_max_iterations = 200 #@param [0, 200, 2000] {type:"raw"}
#@markdown - max amber relax iterations, `0` = unlimited (AlphaFold2 default, can take very long)
#@markdown - 구조 예측 후에 'Amber'라는 물리 기반 정제 과정을 몇 번 반복할지 정하는 설정, 기본값 = `200`
#@markdown - `0`으로 설정하면 반복 횟수 제한 없이 무제한 정제가 가능하지만 실행 시간이 많이 늘어남

pairing_strategy = "greedy" #@param ["greedy", "complete"] {type:"string"}
#@markdown - Multimer 예측에서 여러 사슬 간의 관계를 어떻게 연결할지를 결정하는 전략
#@markdown - `greedy` = pair any taxonomically matching subsets
#@markdown - `complete` = all sequences have to match in one line
#@markdown - `greedy` : 비슷한 종의 서열들끼리 빠르게 짝을 지음, 속도가 빠름
#@markdown - `complete` : 모든 사슬이 정확히 맞는 서열만 쌍으로 만듬, 정밀하긴 하지만 계산 속도가 느림

calc_extra_ptm = False #@param {type:"boolean"}
#@markdown - return pairwise chain iptm/actifptm
#@markdown - 체크하면 ipTM 및 pTM 점수라는 추가적인 구조 신뢰도 지표 제공
#@markdown - 복합체(multimer) 예측 시, 각 체인 간의 상호작용 예측 신뢰도를 수치로 표현
#@markdown - 일반적인 단백질 구조 예측할 때는 체크하지 않아도 되지만, 복합체의 상호작용이 중요한 경우에는 체크

#@markdown ---

#@markdown #### Sample settings
#@markdown -  enable dropouts and increase number of seeds to sample predictions from uncertainty of the model
#@markdown -  decrease `max_msa` to increase uncertainity

max_msa = "auto" #@param ["auto", "512:1024", "256:512", "64:128", "32:64", "16:32"]
#@markdown - MSA의 최대 크기를 설정하는 옵션, 기본값 = `auto`
#@markdown - 값을 줄이면 예측에 활용되는 진화 정보가 적어져서 예측의 불확실성 증가

num_seeds = 1 #@param [1,2,4,8,16] {type:"raw"}
#@markdown - 예측을 시작할 때 사용할 무작위 시드(seed)의 개수
#@markdown - 숫자를 높이면 같은 단백질에 대해 다양한 결과를 얻음, 예측의 일관성이나 변동성을 분석하고 싶을 때 활용

use_dropout = False #@param {type:"boolean"}
#@markdown - 신경망에서 일부 노드를 무작위로 끄는 Dropout 기능을 활성화하는 옵션
#@markdown - Dropout을 켜면 알파폴드가 더 다양한 결과를 생성할 수 있어서 구조 예측의 신뢰도 분포를 평가할 때 도와줌

num_recycles = None if num_recycles == "auto" else int(num_recycles)
recycle_early_stop_tolerance = None if recycle_early_stop_tolerance == "auto" else float(recycle_early_stop_tolerance)
if max_msa == "auto": max_msa = None

#@markdown ---

#@markdown #### Save settings
save_all = False #@param {type:"boolean"}
#@markdown - 체크하면 생성된 **모든 구조 예측 결과(rank 1~5)** 저장

save_recycles = False #@param {type:"boolean"}
#@markdown - 예측 반복(Recycle) 단계의 중간 결과까지 저장하고 싶을 때 사용하는 옵션

save_to_google_drive = False #@param {type:"boolean"}
#@markdown -  if the save_to_google_drive option was selected, the result zip will be uploaded to your Google Drive
#@markdown - 체크하면 결과 파일(zip)이 직접 내 Google Drive에 저장

dpi = 300 #@param {type:"integer"}
#@markdown - set dpi for image resolution
#@markdown - 구조 이미지를 저장할 때 해상도를 설정하는 값

if save_to_google_drive:
  from pydrive2.drive import GoogleDrive
  from pydrive2.auth import GoogleAuth
  from google.colab import auth
  from oauth2client.client import GoogleCredentials
  auth.authenticate_user()
  gauth = GoogleAuth()
  gauth.credentials = GoogleCredentials.get_application_default()
  drive = GoogleDrive(gauth)
  print("You are logged into Google Drive and are good to go!")

#@markdown Don't forget to hit `Runtime` -> `Run all` after updating the form.

In [ ]:
#@title Run Prediction
display_images = True #@param {type:"boolean"}

import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from Bio import BiopythonDeprecationWarning
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type
from colabfold.plot import plot_msa_v2

import os
import numpy as np
try:
  K80_chk = os.popen('nvidia-smi | grep "Tesla K80" | wc -l').read()
except:
  K80_chk = "0"
  pass
if "1" in K80_chk:
  print("WARNING: found GPU Tesla K80: limited to total length < 1000")
  if "TF_FORCE_UNIFIED_MEMORY" in os.environ:
    del os.environ["TF_FORCE_UNIFIED_MEMORY"]
  if "XLA_PYTHON_CLIENT_MEM_FRACTION" in os.environ:
    del os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]

from colabfold.colabfold import plot_protein
from pathlib import Path
import matplotlib.pyplot as plt

# For some reason we need that to get pdbfixer to import
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

def input_features_callback(input_features):
  if display_images:
    plot_msa_v2(input_features)
    plt.show()
    plt.close()

def prediction_callback(protein_obj, length,
                        prediction_result, input_features, mode):
  model_name, relaxed = mode
  if not relaxed:
    if display_images:
      fig = plot_protein(protein_obj, Ls=length, dpi=150)
      plt.show()
      plt.close()

result_dir = jobname
log_filename = os.path.join(jobname,"log.txt")
setup_logging(Path(log_filename))

queries, is_complex = get_queries(queries_path)
model_type = set_model_type(is_complex, model_type)

if "multimer" in model_type and max_msa is not None:
  use_cluster_profile = False
else:
  use_cluster_profile = True

download_alphafold_params(model_type, Path("."))
results = run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=custom_template_path,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,
    num_recycles=num_recycles,
    relax_max_iterations=relax_max_iterations,
    recycle_early_stop_tolerance=recycle_early_stop_tolerance,
    num_seeds=num_seeds,
    use_dropout=use_dropout,
    model_order=[1,2,3,4,5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    pairing_strategy=pairing_strategy,
    stop_at_score=float(100),
    prediction_callback=prediction_callback,
    dpi=dpi,
    zip_results=False,
    save_all=save_all,
    max_msa=max_msa,
    use_cluster_profile=use_cluster_profile,
    input_features_callback=input_features_callback,
    save_recycles=save_recycles,
    user_agent="colabfold/google-colab-main",
    calc_extra_ptm=calc_extra_ptm,
)
results_zip = f"{jobname}.result.zip"
os.system(f"zip -r {results_zip} {jobname}")

In [ ]:
#@title Display 3D structure {run: "auto"}
import py3Dmol
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from colabfold.colabfold import plot_plddt_legend
from colabfold.colabfold import pymol_color_list, alphabet_list

rank_num = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
color = "pLDDT" #@param ["chain", "pLDDT"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

tag = results["rank"][0][rank_num - 1]
jobname_prefix = ".custom" if msa_mode == "custom" else ""
pdb_filename = f"{jobname}/{jobname}{jobname_prefix}_unrelaxed_{tag}.pdb"
pdb_file = glob.glob(pdb_filename)

def show_pdb(rank_num=1, show_sidechains=False, show_mainchains=False, color="lDDT"):
  model_name = f"rank_{rank_num}"
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
  view.addModel(open(pdb_file[0],'r').read(),'pdb')

  if color == "pLDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
  elif color == "chain":
    chains = len(queries[0][1]) + 1 if is_complex else 1
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})

  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  view.zoomTo()
  return view

# 🌟 rank_num 인자를 받도록 함수 업데이트
def custom_plddt_legend(pdb_filepath, rank_num):
    # 1. PDB 파일에서 pLDDT 평균값 계산하기
    plddts = []
    with open(pdb_filepath, 'r') as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                plddts.append(float(line[60:66]))

    # 계산된 평균값 (데이터가 없으면 0 처리)
    avg_plddt = sum(plddts) / len(plddts) if plddts else 0.0

    # 2. 범례 및 텍스트 그리기
    labels = ['Very high (pLDDT > 90)', 'High (90 > pLDDT > 70)', 'Low (70 > pLDDT > 50)', 'Very low (pLDDT < 50)']
    colors = ['blue', 'cyan', 'yellow', 'orange']
    fig = plt.figure(figsize=(14, 1.2), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')

    patches = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, labels)]

    # --- 첫 번째 줄 (상단) ---
    # 범례 타이틀을 'Model confidence:'로 변경하고 상단(y=0.7)에 배치
    ax.text(0.05, 0.7, 'Model confidence:', transform=ax.transAxes, fontsize=12, va='center', ha='left')
    ax.legend(handles=patches, loc='center left', bbox_to_anchor=(0.18, 0.7),
              ncol=4, frameon=False, fontsize=11, columnspacing=1.5)

    # --- 두 번째 줄 (하단) ---
    # 🌟 요청하신 형식 적용: Rank {rank num}: Average pLDDT = XXXX
    formatted_label = f"Rank {rank_num}: Average pLDDT = {avg_plddt:.2f}"

    # 하단 왼쪽(x=0.05, y=0.2)에 굵은 글씨로 정렬하여 배치
    ax.text(0.05, 0.2, formatted_label, transform=ax.transAxes, fontsize=12, va='center', ha='left')

    return fig

# 3D 뷰어 및 올바른 범례 출력
show_pdb(rank_num, show_sidechains, show_mainchains, color).show()
if color == "pLDDT":
    # 🌟 함수 호출 시 pdb_file[0]과 rank_num 2개의 인자를 모두 전달
    custom_plddt_legend(pdb_file[0], rank_num).show()

In [ ]:
#@title Display Average pLDDT {run: "auto"}
import glob
import matplotlib.pyplot as plt
import numpy as np

# Rank별 평균 pLDDT와 라벨을 저장할 리스트
avg_plddts = []
labels = []

# results 딕셔너리에서 생성된 모델 개수만큼 반복 (최대 5개)
num_models = min(5, len(results["rank"][0]))
jobname_prefix = ".custom" if msa_mode == "custom" else ""

for i in range(num_models):
    tag = results["rank"][0][i]
    # PDB 파일 경로 찾기 (unrelaxed 기준)
    pdb_filename = f"{jobname}/{jobname}{jobname_prefix}_unrelaxed_{tag}.pdb"
    pdb_file = glob.glob(pdb_filename)

    if pdb_file:
        # PDB 파일에서 pLDDT 평균값 계산
        plddts = []
        with open(pdb_file[0], 'r') as f:
            for line in f:
                if line.startswith("ATOM") and line[12:16].strip() == "CA":
                    plddts.append(float(line[60:66]))

        avg_plddt = sum(plddts) / len(plddts) if plddts else 0.0
        avg_plddts.append(avg_plddt)
        labels.append(f"Rank {i+1}")

# 그래프 그리기 설정
fig, ax = plt.subplots(figsize=(6, 5), dpi=100)

# 색상 맵 설정
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(avg_plddts)))

# 막대 그래프 생성 (투명도와 테두리 추가)
bars = ax.bar(labels, avg_plddts, color=colors, edgecolor='black', alpha=0.7, zorder=3)

# 배경에 연한 회색 점선 그리드 추가
ax.yaxis.grid(True, linestyle='--', color='lightgray', alpha=0.7, zorder=0)
ax.set_axisbelow(True) # 그리드가 막대 뒤로 가도록 설정

# 각 막대 위에 수치 텍스트 표시
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1, f'{yval:.1f}',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

# 축 및 타이틀 설정
ax.set_ylim(0, 105)
ax.set_ylabel("Average pLDDT") # Y축 라벨
ax.set_title("Average pLDDT per Model", fontweight='bold') # 굵은 타이틀
plt.xticks(rotation=45) # X축 라벨 45도 회전

# 그래프 출력
plt.tight_layout()
plt.show()

In [ ]:
#@title Display PAE plots {run: "auto"}
import json
import os
import glob
import matplotlib.pyplot as plt
import numpy as np

# PAE를 그릴 모델 개수 설정 (최대 5개)
num_models = min(5, len(results["rank"][0]))

# 1행 5열의 서브플롯 생성
fig, axes = plt.subplots(1, 5, figsize=(20, 4), dpi=dpi)

im = None # 하단 컬러바 생성을 위해 마지막 이미지를 저장할 변수

for i in range(5):
    ax = axes[i]
    if i < num_models:
        tag = results["rank"][0][i]

        # 해당 모델의 json 파일 찾기
        json_files = glob.glob(f"{jobname}/*_{tag}*.json")

        pae_matrix = None
        avg_plddt = 0.0

        # JSON 파일에서 pae 데이터와 plddt 점수 가져오기
        for j_file in json_files:
            with open(j_file, 'r') as f:
                data = json.load(f)
                if 'pae' in data:
                    pae_matrix = np.array(data['pae'])
                if 'plddt' in data:
                    avg_plddt = np.mean(data['plddt']) # pLDDT 평균 계산

        if pae_matrix is not None:
            # 진한 초록색이 낮은 에러를 나타내도록 'Greens_r' 사용
            im = ax.imshow(pae_matrix, cmap='Greens_r', vmin=0, vmax=30)

            # 이미지와 동일한 제목 포맷 적용
            ax.set_title(f"Model Rank {i+1} (pLDDT: {avg_plddt:.1f})", fontsize=10)
            ax.set_xlabel("Scored residue")

            # y축 라벨은 첫 번째 그래프에만 표시
            if i == 0:
                ax.set_ylabel("Aligned residue")
        else:
            ax.set_title(f"Model Rank {i+1}\n(Data missing)")
            ax.axis('off')
    else:
        # 5개 미만의 모델인 경우 빈 칸 숨김
        ax.axis('off')

# 전체 타이틀을 좌측 상단에 배치 (사진과 동일)
plt.text(0.01, 1.05, "Predicted Aligned Error (PAE)", transform=fig.transFigure,
         fontsize=14, fontweight='bold', ha='left')

# 이미지처럼 하단 중앙에 가로형 공유 컬러바 1개 추가
if im is not None:
    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), orientation='horizontal',
                        fraction=0.08, aspect=40, pad=0.2)
    cbar.set_label("Expected position error (Ångströms, Å)")

plt.show()

In [ ]:
#@title Display MSA(Multiple Sequence Alignment) coverage plot {run: "auto"}
import matplotlib.image as mpimg
import os
import matplotlib.pyplot as plt

# ColabFold가 이미 만들어둔 coverage 이미지 파일 경로 가져오기
jobname_prefix = ".custom" if msa_mode == "custom" else ""
cov_file = f"{jobname}/{jobname}{jobname_prefix}_coverage.png"

if os.path.exists(cov_file):
    # 그림을 띄울 공간 생성 (적절한 크기 지정)
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

    # 이미지 불러와서 출력
    img = mpimg.imread(cov_file)
    ax.imshow(img)

    # 원본 이미지 자체에 이미 X/Y축과 글씨, 컬러바가 다 포함되어 있으므로
    # matplotlib이 그리는 겉 테두리(축)는 깔끔하게 숨김처리
    ax.axis('off')

    plt.show()
else:
    print(f"MSA 커버리지 이미지를 찾을 수 없습니다: {cov_file}")

In [ ]:
#@title Display per-residue confidence plot {run: "auto"}
import json
import glob
import os
import matplotlib.pyplot as plt

# 그릴 모델 개수 설정 (최대 5개)
num_models = min(5, len(results["rank"][0]))

# 그래프 크기 및 해상도 설정
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)

# 1위부터 5위까지 모델의 JSON 파일을 열어서 pLDDT 데이터 추출
for i in range(num_models):
    tag = results["rank"][0][i]

    # 해당 랭크의 JSON 파일 찾기
    json_files = glob.glob(f"{jobname}/*_{tag}*.json")

    for j_file in json_files:
        with open(j_file, 'r') as f:
            data = json.load(f)

            # 파일 내에 'plddt' 데이터가 있는지 확인
            if 'plddt' in data:
                plddt_scores = data['plddt']

                # 추출한 데이터로 선 그래프 그리기 (X축은 자동으로 0부터 생성됨)
                ax.plot(plddt_scores, label=f"rank_{i+1}")
                break # 데이터 그렸으면 다음 모델로 넘어가기

# 그래프 꾸미기 (타이틀, 축 이름)
ax.set_title("Predicted LDDT(pLDDT) per position", fontsize=14)
ax.set_xlabel("Positions", fontsize=12)
ax.set_ylabel("Predicted lDDT(pLDDT) Score", fontsize=12)

# Y축 범위를 0 ~ 100으로 고정 (사진과 동일하게)
ax.set_ylim(0, 100)

# 좌측 하단에 범례(Legend) 추가
ax.legend(loc="lower left")

# 레이아웃 깔끔하게 정리 후 출력
plt.tight_layout()
plt.show()

In [ ]:
#@title Plots {run: "auto"}
from IPython.display import display, HTML
import base64
from html import escape

# see: https://stackoverflow.com/a/53688522
def image_to_data_url(filename):
  ext = filename.split('.')[-1]
  prefix = f'data:image/{ext};base64,'
  with open(filename, 'rb') as f:
    img = f.read()
  return prefix + base64.b64encode(img).decode('utf-8')

pae = ""
pae_file = os.path.join(jobname,f"{jobname}{jobname_prefix}_pae.png")
if os.path.isfile(pae_file):
    pae = image_to_data_url(pae_file)
cov = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_coverage.png"))
plddt = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_plddt.png"))
display(HTML(f"""
<style>
  img {{
    float:left;
  }}
  .full {{
    max-width:100%;
  }}
  .half {{
    max-width:50%;
  }}
  @media (max-width:640px) {{
    .half {{
      max-width:100%;
    }}
  }}
</style>
<div style="max-width:90%; padding:2em;">
  <h1>Plots for {escape(jobname)}</h1>
  { '<!--' if pae == '' else '' }<img src="{pae}" class="full" />{ '-->' if pae == '' else '' }
  <img src="{cov}" class="half" />
  <img src="{plddt}" class="half" />
</div>
"""))

In [ ]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive, try disabling your adblocker and run this cell again. If that fails click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

if msa_mode == "custom":
  print("Don't forget to cite your custom MSA generation method.")

files.download(f"{jobname}.result.zip")

if save_to_google_drive == True and drive:
  uploaded = drive.CreateFile({'title': f"{jobname}.result.zip"})
  uploaded.SetContentFile(f"{jobname}.result.zip")
  uploaded.Upload()
  print(f"Uploaded {jobname}.result.zip to Google Drive with ID {uploaded.get('id')}")

# Instructions <a name="Instructions"></a>
For detailed instructions, tips and tricks, see recently published paper at [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5)

**Quick start**
1. Paste your protein sequence(s) in the input field.
2. Press "Runtime" -> "Run all".
3. The pipeline consists of 5 steps. The currently running step is indicated by a circle with a stop sign next to it.

**Result zip file contents**

1. PDB formatted structures sorted by avg. pLDDT and complexes are sorted by pTMscore. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. A `predicted_aligned_error_v1.json` using [AlphaFold-DB's format](https://alphafold.ebi.ac.uk/faq#faq-7) and a `scores.json` for each model which contains an array (list of lists) for PAE, a list with the average pLDDT and the pTMscore.
7. BibTeX file with citations for all used tools and databases.

At the end of the job a download modal box will pop up with a `jobname.result.zip` file. Additionally, if the `save_to_google_drive` option was selected, the `jobname.result.zip` will be uploaded to your Google Drive.

**MSA generation for complexes**

For the complex prediction we use unpaired and paired MSAs. Unpaired MSA is generated the same way as for the protein structures prediction by searching the UniRef100 and environmental sequences three iterations each.

The paired MSA is generated by searching the UniRef100 database and pairing the best hits sharing the same NCBI taxonomic identifier (=species or sub-species). We only pair sequences if all of the query sequences are present for the respective taxonomic identifier.

**Using a custom MSA as input**

To predict the structure with a custom MSA (A3M formatted): (1) Change the `msa_mode`: to "custom", (2) Wait for an upload box to appear at the end of the "MSA options ..." box. Upload your A3M. The first fasta entry of the A3M must be the query sequence without gaps.

It is also possilbe to provide custom MSAs for complex predictions. Read more about the format [here](https://github.com/sokrypton/ColabFold/issues/76).

As an alternative for MSA generation the [HHblits Toolkit server](https://toolkit.tuebingen.mpg.de/tools/hhblits) can be used. After submitting your query, click "Query Template MSA" -> "Download Full A3M". Download the A3M file and upload it in this notebook.

**PDB100** <a name="pdb100"></a>

As of 23/06/08, we have transitioned from using the PDB70 to a 100% clustered PDB, the PDB100. The construction methodology of PDB100 differs from that of PDB70.

The PDB70 was constructed by running each PDB70 representative sequence through [HHblits](https://github.com/soedinglab/hh-suite) against the [Uniclust30](https://uniclust.mmseqs.com/). On the other hand, the PDB100 is built by searching each PDB100 representative structure with [Foldseek](https://github.com/steineggerlab/foldseek) against the [AlphaFold Database](https://alphafold.ebi.ac.uk).

To maintain compatibility with older Notebook versions and local installations, the generated files and API responses will continue to be named "PDB70", even though we're now using the PDB100.

**Using custom templates** <a name="custom_templates"></a>

To predict the structure with a custom template (PDB or mmCIF formatted): (1) change the `template_mode` to "custom" in the execute cell and (2) wait for an upload box to appear at the end of the "Input Protein" box. Select and upload your templates (multiple choices are possible).

* Templates must follow the four letter PDB naming with lower case letters.

* Templates in mmCIF format must contain `_entity_poly_seq`. An error is thrown if this field is not present. The field `_pdbx_audit_revision_history.revision_date` is automatically generated if it is not present.

* Templates in PDB format are automatically converted to the mmCIF format. `_entity_poly_seq` and `_pdbx_audit_revision_history.revision_date` are automatically generated.

If you encounter problems, please report them to this [issue](https://github.com/sokrypton/ColabFold/issues/177).

**Comparison to the full AlphaFold2 and AlphaFold2 Colab**

This notebook replaces the homology detection and MSA pairing of AlphaFold2 with MMseqs2. For a comparison against the [AlphaFold2 Colab](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb) and the full [AlphaFold2](https://github.com/deepmind/alphafold) system read our [paper](https://www.nature.com/articles/s41592-022-01488-1).

**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or MGnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses the AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- [KOBIC](https://kobic.re.kr) and [Söding Lab](https://www.mpinat.mpg.de/soeding) for providing the computational resources for the MMseqs2 MSA server.

- Richard Evans for helping to benchmark the ColabFold's Alphafold-multimer support.

- [David Koes](https://github.com/dkoes) for his awesome [py3Dmol](https://3dmol.csb.pitt.edu/) plugin, without whom these notebooks would be quite boring!

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
